In [2]:
import zipfile
import os

zip_path = "Copy of devnagari digit.zip"  # your file name

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [3]:
os.listdir("dataset")

['DevanagariHandwrittenDigitDataset']

In [8]:
# ==========================================
# TASK 2: SET YOUR EXACT DATASET PATH
# ==========================================
train_dir = "dataset/DevanagariHandwrittenDigitDataset/Train/"
test_dir = "dataset/DevanagariHandwrittenDigitDataset/Test/"

print("Train path exists:", os.path.exists(train_dir))
print("Test path exists:", os.path.exists(test_dir))

print("Train classes:", os.listdir(train_dir))
print("Test classes:", os.listdir(test_dir))

Train path exists: True
Test path exists: True
Train classes: ['digit_3', 'digit_8', 'digit_6', 'digit_2', 'digit_7', 'digit_0', 'digit_5', 'digit_9', 'digit_4', 'digit_1']
Test classes: ['digit_3', 'digit_8', 'digit_6', 'digit_2', 'digit_7', 'digit_0', 'digit_5', 'digit_9', 'digit_4', 'digit_1']


In [9]:
# ==========================================
# TASK 3: LOAD AND PREPROCESS IMAGES USING PIL
# ==========================================
img_height = 28
img_width = 28

def load_images_from_folder(folder):
    images = []
    labels = []

    class_names = sorted(os.listdir(folder))
    class_map = {name: i for i, name in enumerate(class_names)}
    print("Class map for", folder, ":", class_map)

    for class_name in class_names:
        class_path = os.path.join(folder, class_name)

        if not os.path.isdir(class_path):
            continue

        label = class_map[class_name]

        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)

            try:
                img = Image.open(img_path).convert("L")      # grayscale
                img = img.resize((img_width, img_height))   # resize to 28x28
                img = np.array(img) / 255.0                 # normalize to [0,1]

                images.append(img)
                labels.append(label)

            except Exception as e:
                print(f"Skipping {img_path} because of error: {e}")

    return np.array(images), np.array(labels)

# Load training and testing data
x_train, y_train = load_images_from_folder(train_dir)
x_test, y_test = load_images_from_folder(test_dir)

print("\nBefore reshape:")
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

Class map for dataset/DevanagariHandwrittenDigitDataset/Train/ : {'digit_0': 0, 'digit_1': 1, 'digit_2': 2, 'digit_3': 3, 'digit_4': 4, 'digit_5': 5, 'digit_6': 6, 'digit_7': 7, 'digit_8': 8, 'digit_9': 9}
Class map for dataset/DevanagariHandwrittenDigitDataset/Test/ : {'digit_0': 0, 'digit_1': 1, 'digit_2': 2, 'digit_3': 3, 'digit_4': 4, 'digit_5': 5, 'digit_6': 6, 'digit_7': 7, 'digit_8': 8, 'digit_9': 9}

Before reshape:
x_train shape: (17000, 28, 28)
y_train shape: (17000,)
x_test shape: (3000, 28, 28)
y_test shape: (3000,)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

# Task 2: Build the FCN Model
model = Sequential([
    Flatten(input_shape=(28, 28, 1)),   # converts 28x28 image into 1D vector
    Dense(64, activation='sigmoid'),    # 1st hidden layer
    Dense(128, activation='sigmoid'),   # 2nd hidden layer
    Dense(256, activation='sigmoid'),   # 3rd hidden layer
    Dense(10, activation='softmax')     # output layer for 10 classes
])

# Show model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,154 (367.79 KB)

 Trainable params: 94,154 (367.79 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Task 3: Compile the Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Callbacks
checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train the model
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=20,
    validation_split=0.2,
    callbacks=[checkpoint, early_stopping]
)

Epoch 1/20
103/107 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2982 - loss: 1.9809
Epoch 1: val_loss improved from None to 7.00918, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.5101 - loss: 1.5941 - val_accuracy: 0.0000e+00 - val_loss: 7.0092
Epoch 2/20
100/107 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8639 - loss: 0.5606
Epoch 2: val_loss did not improve from 7.00918
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8837 - loss: 0.4489 - val_accuracy: 0.0000e+00 - val_loss: 8.2578
Epoch 3/20
103/107 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9264 - loss: 0.2665
Epoch 3: val_loss did not improve from 7.00918
107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9329 - loss: 0.2426 - val_accuracy: 0.0000e+00 - val_loss: 8.8665
Epoch 4/20
100/107 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9507 - loss: 0.1837
Epoch 4: val_loss did not improve from 7.00918
107/107 ━━━━━━━━━

In [14]:
# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=2)

# Print results
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

94/94 - 0s - 3ms/step - accuracy: 0.6740 - loss: 1.9807
Test Loss: 1.9806514978408813
Test Accuracy: 0.6740000247955322


In [ ]:
from tensorflow.keras.models import load_model

# ==============================
# Save the trained model
# ==============================
model.save("devnagari_fcn_model.h5")

print("Model saved successfully!")

# ==============================
# Load the saved model
# ==============================
loaded_model = load_model("devnagari_fcn_model.h5")

print("Model loaded successfully!")

# ==============================
# Re-evaluate loaded model
# ==============================
loaded_loss, loaded_accuracy = loaded_model.evaluate(x_test, y_test, verbose=2)

print("Loaded Model Loss:", loaded_loss)
print("Loaded Model Accuracy:", loaded_accuracy)

In [17]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# Make predictions on test data
# ==============================
predictions = model.predict(x_test)

# ==============================
# Convert probabilities to labels
# ==============================
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test, axis=1)

# ==============================
# Print first few results
# ==============================
print("First 10 Predicted Labels:", predicted_labels[:10])
print("First 10 True Labels:", true_labels[:10])

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
First 10 Predicted Labels: [0 0 0 0 0 0 0 7 0 0]
First 10 True Labels: [0 0 0 0 0 0 0 0 0 0]
